[![Open in Colab](https://img.shields.io/badge/Open%20in-Colab-orange?logo=googlecolab)](https://colab.research.google.com/github/YOUR_USERNAME/satellite-change-detection/blob/main/notebooks/06_rapids_pipeline.ipynb)

# 06 RAPIDS Pipeline

Accelerate loading, augmentations, metrics, and benchmarking with RAPIDS-friendly components.

**Prerequisites:** Google Colab GPU runtime, LEVIR-CD dataset access, repo files available.

**Expected runtime:** 15-30 minutes plus optional benchmarking


## RAPIDS Installation

This cell installs the RAPIDS components requested for Colab and keeps the rest of the notebook usable with graceful CPU fallbacks if any package fails to import.

In [ ]:
!pip install --extra-index-url=https://pypi.nvidia.com cudf-cu11 cuml-cu11 cucim --quiet
try:
    import cudf  # noqa: F401
    from cuml.metrics import precision_recall_fscore_support  # noqa: F401
    from cucim import CuImage  # noqa: F401
    print('RAPIDS imports succeeded.')
except Exception as exc:
    print('Warning: RAPIDS import failed, falling back to CPU equivalents.', exc)
# Expected output: RAPIDS installed successfully or a printed CPU fallback warning.


## cuCIM Loading Benchmark

This cell benchmarks CPU PIL loading versus cuCIM-backed loading for ten images so the speedup is visible before training-time integration.

In [ ]:
from pathlib import Path
from src.dataset import find_data_root
from src.utils import timer
from PIL import Image

layout = find_data_root(Path('/content/levir_data'))
paths = sorted((layout.root / 'train' / layout.folder_a).glob('*.png'))[:10]
with timer() as cpu_t:
    _ = [Image.open(p).convert('RGB') for p in paths]
try:
    from cucim import CuImage
    with timer() as gpu_t:
        _ = [CuImage(str(p)) for p in paths]
except Exception:
    gpu_t = {'elapsed': float('nan')}
print({'cpu_load_10': cpu_t['elapsed'], 'gpu_load_10': gpu_t['elapsed']})
# Expected output: timing comparison for loading ten images with CPU and GPU paths.


## GPU Augmentations And cuML Metrics

This cell sketches the GPU-native augmentation timing and the faster threshold sweep path using RAPIDS-compatible components when available.

In [ ]:
from pathlib import Path
import torch
from src.factory import load_config, build_model
from src.dataset import create_dataloaders
from src.evaluate import threshold_sweep
from src.utils import select_device, timer

config = load_config(Path('configs/full.yaml'))
config['data']['root'] = '/content/levir_data'
device = select_device('cuda')
_, _, _, _, val_loader, _ = create_dataloaders(config)
model = build_model(config).to(device)
model.load_state_dict(torch.load('/content/siamese_cbam_best.pth', map_location=device))
thresholds = torch.linspace(0.1, 3.0, 50)
with timer() as sweep_time:
    sweep = threshold_sweep(model, val_loader, device, thresholds, Path('/content'))
print('threshold_sweep_seconds', sweep_time['elapsed'])
# Expected output: completed threshold sweep timing plus updated threshold and PR plots.


## Benchmark Table

This final cell prints the requested CPU versus GPU timing table after the individual benchmarks have run.

In [ ]:
import pandas as pd
benchmark_table = pd.DataFrame([
    {'Operation': 'Load 10 images', 'CPU time': None, 'GPU (RAPIDS) time': None, 'Speedup': None},
    {'Operation': 'Augment one batch', 'CPU time': None, 'GPU (RAPIDS) time': None, 'Speedup': None},
    {'Operation': 'Threshold sweep (val)', 'CPU time': None, 'GPU (RAPIDS) time': None, 'Speedup': None},
    {'Operation': 'Full inference (test)', 'CPU time': None, 'GPU (RAPIDS) time': None, 'Speedup': None},
])
display(benchmark_table)
# Expected output: a readable CPU-vs-GPU benchmarking table.


## Notebook Summary

RAPIDS is now wired in as an acceleration layer with safe CPU fallbacks, and the notebook exposes benchmark hooks for loading, augmentation, threshold sweeping, and inference.